# EA3 - Actividad 3.3: Dashboard Interactivo en Tiempo Real

## Objetivos
- Visualizar datos de transacciones con widgets interactivos
- Filtrar y explorar datos en vivo usando controles deslizantes y selectores
- Construir un dashboard que permita cambiar parametros sin recargar
- Comparar ventanas temporales con sliders

> **NOTA:** Este notebook funciona con el perfil **basico** de Docker.
> ```bash
> docker-compose --profile basico up -d
> ```
>
> No necesitas Kafka. Usamos datos pre-generados en JSONL.

## Setup

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from datetime import datetime

plt.rcParams['figure.figsize'] = (14, 8)
print("Librerias cargadas.")

# Cargar datos
ruta = "/home/jovyan/datos/streaming/transacciones_1000.jsonl"
if not os.path.exists(ruta):
    print("Generando datos de ejemplo...")
    get_ipython().run_line_magic('run', '/home/jovyan/scripts/generar_datos_streaming.py --tipo transacciones --archivo /home/jovyan/datos/streaming/transacciones_dash.jsonl --cantidad 2000')
    ruta = "/home/jovyan/datos/streaming/transacciones_dash.jsonl"

with open(ruta) as f:
    datos_crudos = [json.loads(linea) for linea in f]

df = pd.DataFrame(datos_crudos)
print(f"Cargados {len(df)} eventos")
print(f"Columnas: {list(df.columns)}")
df.head(3)

## 1. Widgets Interactivos Basicos

Los **ipywidgets** nos permiten crear controles como sliders, dropdowns y botones que actualizan graficos automaticamente.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("ipywidgets disponible.")

### 1.1 Widget Simple: Monto minimo

Un slider que filtra transacciones por monto minimo.

In [ ]:
slider_monto = widgets.IntSlider(
    value=50000,
    min=0,
    max=2000000,
    step=10000,
    description='Monto min:',
    continuous_update=False
)

def filtrar_por_monto(cambio):
    monto_min = slider_monto.value
    filtrados = df[df['monto'] >= monto_min]
    clear_output(wait=True)
    display(slider_monto)
    print(f"\nTransacciones con monto >= ${monto_min:,}: {len(filtrados)}")
    print(f"Monto total: ${filtrados['monto'].sum():,.0f}")
    print(f"Ticket promedio: ${filtrados['monto'].mean():,.0f}")

slider_monto.observe(filtrar_por_monto, names='value')
display(slider_monto)
filtrar_por_monto(None)

## 2. Dashboard Interactivo Completo

Creamos un dashboard con 3 controles:
1. **Dropdown:** Seleccionar metrica a visualizar (monto, cantidad, total)
2. **Slider:** Filtrar por monto minimo
3. **Dropdown:** Agrupar por (producto, metodo_pago, tienda_id)

In [ ]:
def crear_dashboard_interactivo(df):
    """Crea un dashboard con widgets interactivos."""

    # --- Widgets ---
    metrica = widgets.Dropdown(
        options=['monto', 'cantidad', 'total'],
        value='monto',
        description='Metrica:',
    )

    agrupar_por = widgets.Dropdown(
        options=['producto', 'metodo_pago', 'tienda_id', 'region'],
        value='producto',
        description='Agrupar por:',
    )

    monto_min = widgets.IntSlider(
        value=0, min=0, max=2000000, step=50000,
        description='Monto min:',
        continuous_update=False
    )

    top_n = widgets.IntSlider(
        value=10, min=3, max=20, step=1,
        description='Top N:',
        continuous_update=False
    )

    tipo_grafico = widgets.RadioButtons(
        options=['barras', 'barras_h', 'pastel'],
        value='barras_h',
        description='Tipo:',
    )

    # --- Funcion de actualizacion ---
    output = widgets.Output()

    def actualizar_dashboard(*args):
        with output:
            clear_output(wait=True)

            # Filtrar
            dff = df[df['monto'] >= monto_min.value].copy()

            if dff.empty:
                print("Sin datos con ese filtro.")
                return

            # Agrupar
            col = agrupar_por.value
            met = metrica.value
            agg_df = dff.groupby(col)[met].agg(['sum', 'count', 'mean']).reset_index()
            agg_df.columns = [col, 'total', 'conteo', 'promedio']
            agg_df = agg_df.sort_values('total', ascending=False).head(top_n.value)

            # Grafico
            fig, ax = plt.subplots(figsize=(12, 6))
            tip = tipo_grafico.value

            if tip == 'barras':
                ax.bar(agg_df[col].astype(str), agg_df['total'], color='steelblue')
                ax.set_xlabel(col)
                ax.set_ylabel(f'Total {met}')
                plt.xticks(rotation=45, ha='right')
            elif tip == 'barras_h':
                ax.barh(range(len(agg_df)), agg_df['total'], color='steelblue')
                ax.set_yticks(range(len(agg_df)))
                ax.set_yticklabels(agg_df[col].astype(str))
                ax.set_xlabel(f'Total {met}')
            else:
                ax.pie(agg_df['total'], labels=agg_df[col].astype(str),
                       autopct='%1.1f%%', startangle=90)

            titulo = f"{met.upper()} por {col} | Filtro: monto >= ${monto_min.value:,}"
            ax.set_title(titulo, fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()

            # Tabla resumen
            print(f"\nEstadisticas ({len(dff)} transacciones):")
            print(f"  Total {met}: ${dff[met].sum():,.0f}")
            print(f"  Promedio: ${dff[met].mean():,.0f}")
            print(f"  Min: ${dff[met].min():,} | Max: ${dff[met].max():,}")

    # --- Conectar widgets ---
    metrica.observe(actualizar_dashboard, names='value')
    agrupar_por.observe(actualizar_dashboard, names='value')
    monto_min.observe(actualizar_dashboard, names='value')
    top_n.observe(actualizar_dashboard, names='value')
    tipo_grafico.observe(actualizar_dashboard, names='value')

    # --- Layout ---
    controles = widgets.VBox([
        widgets.HBox([metrica, agrupar_por, tipo_grafico]),
        widgets.HBox([monto_min, top_n]),
    ])

    dashboard = widgets.VBox([controles, output])

    # Mostrar inicial
    actualizar_dashboard()

    return dashboard


dashboard = crear_dashboard_interactivo(df)
display(dashboard)

### ?Que Acaba de Pasar?

Creamos un dashboard interactivo con:
- **Selector de metrica:** Cambia entre monto, cantidad y total
- **Selector de agrupacion:** Producto, metodo de pago, tienda o region
- **Filtro de monto minimo:** Un slider que filtra al instante
- **Top N:** Cuantos elementos mostrar
- **Tipo de grafico:** Barras, barras horizontales o pastel

Todo se actualiza **automaticamente** al cambiar cualquier control.

## 3. Dashboard de Ventana Temporal

Ahora creamos un dashboard que simula **ventanas temporales** y muestra como cambian las metricas segun el tamano de la ventana.

In [ ]:
def crear_dashboard_ventanas(df):
    """Dashboard que muestra el efecto del tamano de ventana."""

    # Widgets
    ventana_size = widgets.IntSlider(
        value=50, min=10, max=200, step=10,
        description='Eventos/ventana:',
        continuous_update=False
    )

    metrica_ventana = widgets.Dropdown(
        options=['monto', 'cantidad', 'total'],
        value='monto',
        description='Metrica:',
    )

    output_v = widgets.Output()

    def actualizar_ventanas(*args):
        with output_v:
            clear_output(wait=True)

            tam = ventana_size.value
            met = metrica_ventana.value

            # Dividir en ventanas
            n_ventanas = len(df) // tam
            ventanas = []

            for i in range(n_ventanas):
                inicio = i * tam
                fin = inicio + tam
                ventana_df = df.iloc[inicio:fin]
                ventanas.append({
                    'ventana': i + 1,
                    'sum': ventana_df[met].sum(),
                    'mean': ventana_df[met].mean(),
                    'count': len(ventana_df),
                    'std': ventana_df[met].std(),
                })

            df_v = pd.DataFrame(ventanas)

            # Grafico
            fig, axes = plt.subplots(1, 3, figsize=(16, 5))

            # 1: Suma por ventana
            axes[0].plot(df_v['ventana'], df_v['sum'], 'o-', color='steelblue',
                        linewidth=2, markersize=6)
            axes[0].axhline(y=df_v['sum'].mean(), color='red', linestyle='--',
                          label=f'Promedio: ${df_v["sum"].mean():,.0f}')
            axes[0].fill_between(df_v['ventana'], df_v['sum'] - df_v['std'],
                               df_v['sum'] + df_v['std'], alpha=0.1, color='steelblue')
            axes[0].set_title(f'Total {met} por Ventana ({tam} eventos c/u)')
            axes[0].set_xlabel('Ventana #')
            axes[0].set_ylabel(f'Total {met}')
            axes[0].legend()

            # 2: Promedio por ventana
            axes[1].bar(df_v['ventana'], df_v['mean'], color='coral', alpha=0.7)
            axes[1].axhline(y=df_v['mean'].mean(), color='red', linestyle='--',
                          label=f'Promedio global: ${df_v["mean"].mean():,.0f}')
            axes[1].set_title(f'Promedio de {met} por Ventana')
            axes[1].set_xlabel('Ventana #')
            axes[1].set_ylabel(f'Promedio {met}')
            axes[1].legend()

            # 3: Histograma de montos por ventana
            axes[2].hist(df_v['sum'], bins=15, color='steelblue', edgecolor='white', alpha=0.7)
            axes[2].axvline(x=df_v['sum'].mean(), color='red', linestyle='--',
                          label=f'Media: ${df_v["sum"].mean():,.0f}')
            axes[2].set_title(f'Distribucion de {met} por Ventana')
            axes[2].set_xlabel(f'Total {met}')
            axes[2].set_ylabel('Frecuencia')
            axes[2].legend()

            plt.suptitle(f'Efecto del Tamano de Ventana ({tam} eventos)',
                        fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

            # Estadisticas
            print(f"\nResumen de {n_ventanas} ventanas de {tam} eventos:")
            print(f"  Total {met} promedio por ventana: ${df_v['sum'].mean():,.0f}")
            print(f"  Desviacion estandar: ${df_v['sum'].std():,.0f}")
            print(f"  Min: ${df_v['sum'].min():,} | Max: ${df_v['sum'].max():,}")
            print(f"  Coeficiente de variacion: {df_v['sum'].std() / df_v['sum'].mean() * 100:.1f}%")

    ventana_size.observe(actualizar_ventanas, names='value')
    metrica_ventana.observe(actualizar_ventanas, names='value')

    controles = widgets.HBox([ventana_size, metrica_ventana])
    dashboard_v = widgets.VBox([controles, output_v])

    actualizar_ventanas()
    return dashboard_v


display(crear_dashboard_ventanas(df))

### Preguntas:

1. ?Que pasa con la desviacion estandar cuando aumentas el tamano de ventana?
2. ?Por que el promedio por ventana se acerca mas al promedio global con ventanas grandes?
3. ?Que relacion tiene esto con el **Teorema Central del Limite**?

## 4. Dashboard Comparativo: Tipos de Datos Streaming

Ahora exploramos los diferentes tipos de datos disponibles en `datos/streaming/`.

In [ ]:
def crear_explorador_streaming():
    """Selector de archivos streaming con resumen automatico."""

    ruta_base = "/home/jovyan/datos/streaming"
    archivos_jsonl = [f for f in os.listdir(ruta_base) if f.endswith('.jsonl')]

    selector = widgets.Dropdown(
        options=archivos_jsonl,
        value=archivos_jsonl[0] if archivos_jsonl else None,
        description='Archivo:',
    )

    output_e = widgets.Output()

    def explorar(*args):
        with output_e:
            clear_output(wait=True)

            if not selector.value:
                print("No hay archivos disponibles.")
                return

            archivo = os.path.join(ruta_base, selector.value)

            with open(archivo) as f:
                datos = [json.loads(linea) for linea in f]

            df_ex = pd.DataFrame(datos)

            print(f"Archivo: {selector.value}")
            print(f"Eventos: {len(df_ex):,}")
            print(f"Columnas: {list(df_ex.columns)}")
            print(f"Tipo evento: {df_ex.get('tipo_evento', df_ex.get('tipo', 'N/A')).iloc[0]}")
            print()

            # Columnas numericas
            num_cols = df_ex.select_dtypes(include='number').columns
            if len(num_cols) > 0:
                print("Estadisticas columnas numericas:")
                print(df_ex[num_cols].describe().to_string())

            print("\nPrimeros 3 registros:")
            print(df_ex.head(3).to_string())

    selector.observe(explorar, names='value')
    explorar()

    return widgets.VBox([selector, output_e])


display(crear_explorador_streaming())

## 5. Simulacion de Llegada en Tiempo Real con Widgets

Combinamos todo: un boton que "consume" eventos uno por uno del archivo JSONL y actualiza las metricas en vivo, como si llegaran desde Kafka.

In [ ]:
class SimuladorTiempoReal:
    """Simula la llegada de eventos uno por uno con widgets."""

    def __init__(self, df):
        self.df = df
        self.idx = 0
        self.monto_acum = 0
        self.conteo = 0
        self.montos = []
        self.eventos_por_producto = defaultdict(int)

        # Widgets
        self.btn_siguiente = widgets.Button(description='> Siguiente evento',
                                            button_style='primary',
                                            layout=widgets.Layout(width='200px'))
        self.btn_auto = widgets.Button(description='> Reproducir 10',
                                       button_style='info',
                                       layout=widgets.Layout(width='200px'))
        self.btn_reset = widgets.Button(description='Reiniciar',
                                        button_style='danger',
                                        layout=widgets.Layout(width='200px'))
        self.label_estado = widgets.HTML(value="Listo. Presiona 'Siguiente' para comenzar.")
        self.output_sim = widgets.Output()

        # Conectar
        self.btn_siguiente.on_click(self.siguiente_evento)
        self.btn_auto.on_click(self.reproducir_10)
        self.btn_reset.on_click(self.reiniciar)

    def siguiente_evento(self, btn=None):
        if self.idx >= len(self.df):
            self.label_estado.value = "No hay mas eventos."
            return

        evento = self.df.iloc[self.idx]
        self.idx += 1
        self.conteo += 1
        monto = evento.get('monto', 0) or 0
        self.monto_acum += monto
        self.montos.append(monto)
        self.eventos_por_producto[evento.get('producto', '?')] += 1

        self.actualizar_vista()

    def reproducir_10(self, btn=None):
        for _ in range(10):
            self.siguiente_evento()

    def reiniciar(self, btn=None):
        self.idx = 0
        self.monto_acum = 0
        self.conteo = 0
        self.montos = []
        self.eventos_por_producto.clear()
        self.label_estado.value = "Reiniciado. Presiona 'Siguiente'."
        self.output_sim.clear_output(wait=True)

    def actualizar_vista(self):
        with self.output_sim:
            clear_output(wait=True)

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            # Evento actual
            ultimo = self.df.iloc[self.idx - 1]
            self.label_estado.value = (
                f"<b>Evento #{self.idx}:</b> {ultimo.get('producto','?')} | "
                f"${ultimo.get('monto',0):,.0f} | "
                f"{ultimo.get('metodo_pago','?')}"
            )

            # Grafico 1: Monto acumulado
            axes[0].plot(range(1, len(self.montos) + 1),
                       np.cumsum(self.montos), 'b-', linewidth=2)
            axes[0].fill_between(range(1, len(self.montos) + 1),
                                np.cumsum(self.montos), alpha=0.1, color='blue')
            axes[0].set_title('Monto Acumulado')
            axes[0].set_xlabel('Evento #')
            axes[0].set_ylabel('Total ($)')

            # Grafico 2: Productos
            if self.eventos_por_producto:
                prods = list(self.eventos_por_producto.keys())[:8]
                vals = [self.eventos_por_producto[p] for p in prods]
                axes[1].barh(prods, vals, color='coral')
            axes[1].set_title('Productos (top 8)')
            axes[1].set_xlabel('Conteo')

            # Grafico 3: Metricas
            axes[2].axis('off')
            prom = self.monto_acum / self.conteo if self.conteo > 0 else 0
            texto = (
                f"Procesados:\n{self.conteo} eventos\n\n"
                f"Monto total:\n${self.monto_acum:,.0f}\n\n"
                f"Promedio:\n${prom:,.0f}\n\n"
                f"Restantes:\n{len(self.df) - self.idx}"
            )
            axes[2].text(0.5, 0.5, texto,
                       transform=axes[2].transAxes,
                       fontsize=14, ha='center', va='center',
                       bbox=dict(boxstyle='round', facecolor='lightyellow'))
            axes[2].set_title('Metricas')

            plt.tight_layout()
            plt.show()

    def mostrar(self):
        controles = widgets.HBox([
            self.btn_siguiente,
            self.btn_auto,
            self.btn_reset,
        ])
        return widgets.VBox([
            self.label_estado,
            controles,
            self.output_sim,
        ])


import numpy as np
simulador = SimuladorTiempoReal(df)
display(simulador.mostrar())

---
## Ejercicios

In [ ]:
# =============================================================
# EJERCICIO 1: Dashboard de logs
# =============================================================
# TODO: Carga logs_1000.jsonl y crea un dashboard que permita:
#
# 1. Seleccionar un endpoint especifico (dropdown)
# 2. Ver la distribucion de status_code (grafico de barras)
# 3. Mostrar el response_time promedio como metrica
#
# Pistas:
#   - Carga el archivo: with open(ruta) as f: datos = [json.loads(l) for l in f]
#   - Crea un DataFrame con pd.DataFrame(datos)
#   - Usa widgets.Dropdown para seleccionar endpoint
#   - Agrupa con: df[df['endpoint'] == seleccion]['status_code'].value_counts()

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Dashboard de sensores IoT
# =============================================================
# TODO: Carga iot_1000.jsonl y crea un dashboard con:
#
# 1. Dropdown para seleccionar ubicacion
# 2. Slider para filtrar por nivel de bateria minimo
# 3. Grafico de dispersion: temperatura vs humedad
#    (cada punto es una lectura, coloreado por ubicacion)
# 4. Mostrar cuantas lecturas hay y cuantas tienen bateria baja (< 30%)
#
# Pistas:
#   - dispersion: axes.scatter(x, y, c=colores, alpha=0.6)
#   - Filtro: df[df['bateria'] >= bateria_min]
#   - Bateria baja: df[df['bateria'] < 30]

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Dashboard de stock market
# =============================================================
# TODO: Carga stock_1000.jsonl y crea:
#
# 1. Dropdown para seleccionar simbolo (AAPL, GOOGL, etc.)
# 2. Slider para el numero de ticks a mostrar
# 3. Grafico de linea: precio a traves del tiempo
# 4. Mostrar: precio actual, cambio % diario, volumen promedio
#
# Pistas:
#   - Filtra: df_stock[df_stock['simbolo'] == seleccion]
#   - Toma los ultimos N: .tail(n)
#   - Grafico: ax.plot(precios, 'o-', linewidth=2)

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **ipywidgets:** Sliders, dropdowns, botones y radiobuttons para interactividad
2. **Dashboard filtrable:** Controles que actualizan graficos automaticamente
3. **Ventanas temporales:** Como el tamano de ventana afecta las metricas
4. **Explorador de datos:** Selector de archivos para diferentes tipos de eventos
5. **Simulacion de llegada:** Boton por boton, como si los eventos llegaran de Kafka
6. **Patron observer:** Los widgets "observan" cambios y reaccionan automaticamente

---
## Desafio Extra (Opcional)

**Dashboard completo con tabs:**

Usa `widgets.Tab` para crear un dashboard con 3 pestanas:
1. **Transacciones:** Dashboard de transacciones con filtros
2. **Logs:** Monitoreo de logs web
3. **IoT:** Sensores en tiempo real

Cada pestana carga su propio archivo JSONL y tiene sus propios controles.

In [ ]:
# =============================================================
# DESAFIO: Dashboard multi-pestana
# =============================================================
# TODO: Crea un dashboard con widgets.Tab que tenga 3 pestanas:
#
# 1. "Transacciones" - Filtros: monto min, producto, metodo de pago
# 2. "Logs Web" - Filtros: endpoint, status code
# 3. "Sensores IoT" - Filtros: ubicacion, bateria min
#
# Pistas:
#   - tab = widgets.Tab(children=[pestana1, pestana2, pestana3])
#   - tab.set_title(0, 'Transacciones')
#   - Cada pestana es un widgets.VBox con sus controles + output

# Escribe tu codigo aqui:
